In [ ]:
import pandas as pd
import numpy as np


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re


In [14]:
book_tags = pd.read_csv('../data/book_tags.csv')
books = pd.read_csv('../data/books.csv')
ratings = pd.read_csv('../data/ratings.csv')
tags = pd.read_csv('../data/tags.csv')
to_read = pd.read_csv('../data/to_read.csv')

In [57]:
book_tags

,goodreads_book_id,tag_id,count
0,1,30574,167697
1,1,11305,37174
2,1,11557,34173
3,1,8717,12986
4,1,33114,12716
...,...,...,...
999907,33288638,21303,7
999908,33288638,17271,7
999909,33288638,1126,7
999910,33288638,11478,7


In [17]:
books.columns

Index(['id', 'book_id', 'best_book_id', 'work_id', 'books_count', 'isbn',
       'isbn13', 'authors', 'original_publication_year', 'original_title',
       'title', 'language_code', 'average_rating', 'ratings_count',
       'work_ratings_count', 'work_text_reviews_count', 'ratings_1',
       'ratings_2', 'ratings_3', 'ratings_4', 'ratings_5', 'image_url',
       'small_image_url'],
      dtype='object')

In [5]:
ratings

,book_id,user_id,rating
0,1,314,5
1,1,439,3
2,1,588,5
3,1,1169,4
4,1,1185,4
...,...,...,...
981751,10000,48386,5
981752,10000,49007,4
981753,10000,49383,5
981754,10000,50124,5


In [22]:
tags

,tag_id,tag_name
0,0,-
1,1,--1-
2,2,--10-
3,3,--12-
4,4,--122-
...,...,...
34247,34247,Ｃhildrens
34248,34248,Ｆａｖｏｒｉｔｅｓ
34249,34249,Ｍａｎｇａ
34250,34250,ＳＥＲＩＥＳ


In [44]:
to_read

,user_id,book_id
0,1,112
1,1,235
2,1,533
3,1,1198
4,1,1874
...,...,...
912700,53424,4716
912701,53424,4844
912702,53424,5907
912703,53424,7569


In [45]:
to_read['user_id'].value_counts().mean()


np.float64(18.675799553927686)

In [42]:
book_tags[book_tags['goodreads_book_id'] == 1]

,goodreads_book_id,tag_id,count
0,1,30574,167697
1,1,11305,37174
2,1,11557,34173
3,1,8717,12986
4,1,33114,12716
...,...,...,...
95,1,5444,266
96,1,18913,265
97,1,11595,264
98,1,6829,263


In [46]:
tags

,tag_id,tag_name
0,0,-
1,1,--1-
2,2,--10-
3,3,--12-
4,4,--122-
...,...,...
34247,34247,Ｃhildrens
34248,34248,Ｆａｖｏｒｉｔｅｓ
34249,34249,Ｍａｎｇａ
34250,34250,ＳＥＲＩＥＳ


# Tag names filtering to only genres

In [48]:
book_tags_named = book_tags.merge(tags, on='tag_id', how='inner')
book_tags_named

,goodreads_book_id,tag_id,count,tag_name
0,1,30574,167697,to-read
1,1,11305,37174,fantasy
2,1,11557,34173,favorites
3,1,8717,12986,currently-reading
4,1,33114,12716,young-adult
...,...,...,...,...
999907,33288638,21303,7,neighbors
999908,33288638,17271,7,kindleunlimited
999909,33288638,1126,7,5-star-reads
999910,33288638,11478,7,fave-author


In [71]:
top_tag_names = book_tags_named.groupby('tag_name')['count'].sum().sort_values(ascending=False)
top_tag_names.head(20)

tag_name
to-read               140718761
currently-reading       7507958
favorites               4503173
fiction                 3688819
fantasy                 3548157
young-adult             1848306
classics                1756920
books-i-own             1317235
romance                 1231926
owned                   1224279
ya                       898334
mystery                  872282
non-fiction              857901
historical-fiction       815421
series                   782637
science-fiction          703866
sci-fi                   597325
paranormal               542559
kindle                   506882
contemporary             486001
Name: count, dtype: int64

In [52]:
synonym_map = {
    'ya': 'young-adult',
    'sci-fi': 'science-fiction',
    'sci-fi-fantasy': 'science-fiction',
    'nonfiction': 'non-fiction',
    'favourites': 'favorites',
    'classic': 'classics',
    'graphic-novel': 'graphic-novels',
    'audiobooks': 'audiobook',
    'children': 'childrens',
    'children-s': 'childrens',
    'children-s-books': 'childrens',
    'picture-books': 'childrens',
    'ebooks': 'ebook',
    'novel': 'novels',
    'adult-fiction': 'adult',
}

book_tags_named['tag_name_clean'] = book_tags_named['tag_name'].replace(synonym_map)

top_tag_names = book_tags_named.groupby('tag_name_clean')['count'].sum().sort_values(ascending=False)

In [53]:
top_tag_names

tag_name_clean
to-read                    140718761
currently-reading            7507958
favorites                    4926462
fiction                      3688819
fantasy                      3548157
                             ...    
0-a-find-2016-summer-00            1
0-all2                             1
0-best-picture-younger             1
challenge-2013                    -1
america-in-retreat                -1
Name: count, Length: 34237, dtype: int64

In [54]:
genre_whitelist = {
    'fiction','fantasy','young-adult','classics','romance','mystery','non-fiction',
    'historical-fiction','science-fiction','paranormal','contemporary','horror',
    'childrens','thriller','history','dystopian','humor','literature','dystopia',
    'crime','graphic-novels','memoir','biography','philosophy','poetry','manga',
    'suspense','new-adult','drama','plays','short-stories','middle-grade',
    'mythology','psychology','business','religion','christian','erotica',
    'self-help','politics','war','coming-of-age','action','comedy','family',
    'vampires','supernatural','magic','adventure','teen','school',
}

genre_only = book_tags_named[book_tags_named['tag_name_clean'].isin(genre_whitelist)]
genre_totals = genre_only.groupby('tag_name_clean')['count'].sum().sort_values(ascending=False)

In [56]:
genre_only

,goodreads_book_id,tag_id,count,tag_name,tag_name_clean
1,1,11305,37174,fantasy,fantasy
4,1,33114,12716,young-adult,young-adult
5,1,11743,9954,fiction,fiction
9,1,32989,4364,ya,young-adult
12,1,18886,3374,magic,magic
...,...,...,...,...,...
999842,33288638,11743,21,fiction,fiction
999858,33288638,6857,14,children,childrens
999864,33288638,11221,13,family,family
999869,33288638,15048,12,humor,humor


# Join to get title name and aggregate all version

In [58]:
genre_with_titles = genre_only.merge(
    books[['book_id', 'title']], left_on='goodreads_book_id', right_on='book_id', how='inner'
)

# aggregate all editions/versions under the same title
title_genre_counts = (
    genre_with_titles.groupby(['title', 'tag_name_clean'])['count']
    .sum()
    .reset_index()
)

In [84]:
n = 5

top_n_genres = (
    title_genre_counts
    .sort_values(['title', 'count'], ascending=[True, False])
    .groupby('title')
    .head(n)
)

In [88]:
genre_combined = (
    top_n_genres
    .assign(token=top_n_genres['tag_name_clean'].str.replace('-', '_'))
    .groupby('title')['token']
    .apply(lambda toks: ' '.join(toks))
    .reset_index()
    .rename(columns={'token': 'genre_combined'})
)

In [89]:
genre_combined

,title,genre_combined
0,"Angels (Walsh Family, #3)",fiction romance contemporary humor drama
1,"""حكايات فرغلي المستكاوي ""حكايتى مع كفر السحلاوية",short_stories horror comedy fiction humor
2,#GIRLBOSS,non_fiction memoir business self_help biography
3,'Salem's Lot,horror vampires thriller fiction paranormal
4,"'Tis (Frank McCourt, #2)",non_fiction memoir biography fiction history
...,...,...
9959,واحة الغروب,fiction historical_fiction literature history ...
9960,يوتوبيا,dystopia fiction science_fiction literature fa...
9961,ڤيرتيجو,fiction crime literature thriller mystery
9962,キスよりも早く1 [Kisu Yorimo Hayaku 1] (Faster than a...,romance manga graphic_novels young_adult conte...


# total average across all books with same title

In [77]:
title_rating_totals = books.groupby('title')[['ratings_1','ratings_2','ratings_3','ratings_4','ratings_5']].sum().reset_index()

title_rating_totals['total_ratings'] = title_rating_totals[['ratings_1','ratings_2','ratings_3','ratings_4','ratings_5']].sum(axis=1)
title_rating_totals['avg_rating'] = (
    title_rating_totals['ratings_1']*1 +
    title_rating_totals['ratings_2']*2 +
    title_rating_totals['ratings_3']*3 +
    title_rating_totals['ratings_4']*4 +
    title_rating_totals['ratings_5']*5
) / title_rating_totals['total_ratings']

# Buckets for books one for each decade based on most popular adaptation

In [81]:
def to_decade(year):
    if pd.isna(year):
        return 'unknown'
    year = int(year)
    decade_start = (year // 10) * 10
    return f'{decade_start}s'

books['decade'] = books['original_publication_year'].apply(to_decade)

idx = books.groupby('title')['ratings_count'].idxmax()
title_popular_year = books.loc[idx, ['title', 'original_publication_year', 'ratings_count']].reset_index(drop=True)
title_popular_year['decade'] = title_popular_year['original_publication_year'].apply(to_decade)

In [82]:
title_popular_year

,title,original_publication_year,ratings_count,decade
0,"Angels (Walsh Family, #3)",2002.0,25680,2000s
1,"""حكايات فرغلي المستكاوي ""حكايتى مع كفر السحلاوية",2013.0,7443,2010s
2,#GIRLBOSS,2014.0,40090,2010s
3,'Salem's Lot,1975.0,228680,1970s
4,"'Tis (Frank McCourt, #2)",1999.0,40726,1990s
...,...,...,...,...
9959,واحة الغروب,2006.0,10365,2000s
9960,يوتوبيا,2008.0,31669,2000s
9961,ڤيرتيجو,2007.0,17001,2000s
9962,キスよりも早く1 [Kisu Yorimo Hayaku 1] (Faster than a...,2007.0,11232,2000s


# Author most popular adaptation

In [90]:
books['author_token'] = (
    books['authors']
    .str.split(',').str[0]
    .str.strip()
    .str.replace(' ', '_')
    .str.replace('.', '', regex=False)
)

idx = books.groupby('title')['ratings_count'].idxmax()
title_popular_author = books.loc[idx, ['title', 'author_token', 'ratings_count']].reset_index(drop=True)

In [91]:
title_popular_author

,title,author_token,ratings_count
0,"Angels (Walsh Family, #3)",Marian_Keyes,25680
1,"""حكايات فرغلي المستكاوي ""حكايتى مع كفر السحلاوية",حسن_الجندي,7443
2,#GIRLBOSS,Sophia_Amoruso,40090
3,'Salem's Lot,Stephen_King,228680
4,"'Tis (Frank McCourt, #2)",Frank_McCourt,40726
...,...,...,...
9959,واحة الغروب,بهاء_طاهر,10365
9960,يوتوبيا,أحمد_خالد_توفيق,31669
9961,ڤيرتيجو,أحمد_مراد,17001
9962,キスよりも早く1 [Kisu Yorimo Hayaku 1] (Faster than a...,Meca_Tanaka,11232


# Language based on most popular adaptation

In [ ]:
lang_synonym_map = {'en-US': 'eng', 'en-GB': 'eng', 'en-CA': 'eng', 'en': 'eng'}
books['language_code_clean'] = books['language_code'].replace(lang_synonym_map)
books['language_code_clean'] = books['language_code_clean'].fillna('unknown')

idx = books.groupby('title')['ratings_count'].idxmax()
title_popular_language = books.loc[idx, ['title', 'language_code_clean', 'ratings_count']].reset_index(drop=True)

# Most popular adaptaion combined

there are 33 duplicates, different versions of the same book, this just solves that by using most popular version

In [99]:
idx = books.groupby('title')['ratings_count'].idxmax()
title_popular_edition = books.loc[idx, [
    'id', 'title', 'author_token', 'original_publication_year', 'language_code_clean', 'ratings_count'
]].reset_index(drop=True)
title_popular_edition['decade'] = title_popular_edition['original_publication_year'].apply(to_decade)

title_popular_edition

,id,title,author_token,original_publication_year,language_code_clean,ratings_count,decade
0,3998,"Angels (Walsh Family, #3)",Marian_Keyes,2002.0,eng,25680,2000s
1,9610,"""حكايات فرغلي المستكاوي ""حكايتى مع كفر السحلاوية",حسن_الجندي,2013.0,ara,7443,2010s
2,2855,#GIRLBOSS,Sophia_Amoruso,2014.0,eng,40090,2010s
3,349,'Salem's Lot,Stephen_King,1975.0,eng,228680,1970s
4,2252,"'Tis (Frank McCourt, #2)",Frank_McCourt,1999.0,eng,40726,1990s
...,...,...,...,...,...,...,...
9959,8247,واحة الغروب,بهاء_طاهر,2006.0,ara,10365,2000s
9960,2588,يوتوبيا,أحمد_خالد_توفيق,2008.0,ara,31669,2000s
9961,3538,ڤيرتيجو,أحمد_مراد,2007.0,ara,17001,2000s
9962,9321,キスよりも早く1 [Kisu Yorimo Hayaku 1] (Faster than a...,Meca_Tanaka,2007.0,jpn,11232,2000s


In [105]:
genre_combined_full = genre_combined.merge(title_popular_edition, on='title', how='left')
genre_combined_full = genre_combined_full.set_index('id')

In [106]:
genre_combined_full

,title,genre_combined,author_token,original_publication_year,language_code_clean,ratings_count,decade
id,,,,,,,
3998,"Angels (Walsh Family, #3)",fiction romance contemporary humor drama,Marian_Keyes,2002.0,eng,25680,2000s
9610,"""حكايات فرغلي المستكاوي ""حكايتى مع كفر السحلاوية",short_stories horror comedy fiction humor,حسن_الجندي,2013.0,ara,7443,2010s
2855,#GIRLBOSS,non_fiction memoir business self_help biography,Sophia_Amoruso,2014.0,eng,40090,2010s
349,'Salem's Lot,horror vampires thriller fiction paranormal,Stephen_King,1975.0,eng,228680,1970s
2252,"'Tis (Frank McCourt, #2)",non_fiction memoir biography fiction history,Frank_McCourt,1999.0,eng,40726,1990s
...,...,...,...,...,...,...,...
8247,واحة الغروب,fiction historical_fiction literature history ...,بهاء_طاهر,2006.0,ara,10365,2000s
2588,يوتوبيا,dystopia fiction science_fiction literature fa...,أحمد_خالد_توفيق,2008.0,ara,31669,2000s
3538,ڤيرتيجو,fiction crime literature thriller mystery,أحمد_مراد,2007.0,ara,17001,2000s


In [103]:
# map every individual book id -> its title -> canonical (most popular) id
id_to_title = books.set_index('id')['title'].to_dict()
title_to_canonical_id = title_popular_edition.set_index('title')['id'].to_dict()

ratings['title'] = ratings['book_id'].map(id_to_title)
ratings['canonical_id'] = ratings['title'].map(title_to_canonical_id)

ratings

,book_id,user_id,rating,title,canonical_id
0,1,314,5,"The Hunger Games (The Hunger Games, #1)",1
1,1,439,3,"The Hunger Games (The Hunger Games, #1)",1
2,1,588,5,"The Hunger Games (The Hunger Games, #1)",1
3,1,1169,4,"The Hunger Games (The Hunger Games, #1)",1
4,1,1185,4,"The Hunger Games (The Hunger Games, #1)",1
...,...,...,...,...,...
981751,10000,48386,5,The First World War,10000
981752,10000,49007,4,The First World War,10000
981753,10000,49383,5,The First World War,10000
981754,10000,50124,5,The First World War,10000


In [109]:
to_read['title'] = to_read['book_id'].map(id_to_title)
to_read['canonical_id'] = to_read['title'].map(title_to_canonical_id)

to_read_clean = to_read.drop_duplicates(subset=['user_id', 'canonical_id'])

to_read_clean

,user_id,book_id,title,canonical_id
0,1,112,"Me Before You (Me Before You, #1)",112
1,1,235,The Husband's Secret,235
2,1,533,Go Set a Watchman,533
3,1,1198,A Little Life,1198
4,1,1874,The Aviator's Wife,1874
...,...,...,...,...
912700,53424,4716,Kim,4716
912701,53424,4844,"Half Magic (Tales of Magic, #1)",4844
912702,53424,5907,Our Mutual Friend,5907
912703,53424,7569,"An Army at Dawn: The War in North Africa, 1942...",7569


# remove any to read books if the user already read it

In [111]:
to_read_filtered = to_read_clean.merge(
    ratings[['user_id', 'canonical_id']].drop_duplicates(),
    on=['user_id', 'canonical_id'],
    how='left',
    indicator=True
)
to_read_filtered = to_read_filtered[to_read_filtered['_merge'] == 'left_only'].drop(columns='_merge')

In [112]:
to_read_filtered

,user_id,book_id,title,canonical_id
0,1,112,"Me Before You (Me Before You, #1)",112
1,1,235,The Husband's Secret,235
2,1,533,Go Set a Watchman,533
3,1,1198,A Little Life,1198
4,1,1874,The Aviator's Wife,1874
...,...,...,...,...
912649,53424,4716,Kim,4716
912650,53424,4844,"Half Magic (Tales of Magic, #1)",4844
912651,53424,5907,Our Mutual Friend,5907
912652,53424,7569,"An Army at Dawn: The War in North Africa, 1942...",7569
